# ☁️ Antigravity Sovereign Linter Worker (Colab Extension)
This Colab notebook connects to your Google Drive VFS to continuously lint, refactor, and fix spelling/syntax errors in the Monorepo without eating up your local Mac's compute.

In [ ]:
from google.colab import drive

# Mount the VFS
drive.mount("/content/drive")

# Adjust this path if your monorepo is located elsewhere in your Google Drive
MONOREPO_DIR = "/content/drive/MyDrive/Monorepo-Uphillsnowball"
print(f"Targeting Monorepo at: {MONOREPO_DIR}")

In [ ]:
!echo "Installing heavy linting and refactoring toolchains..."
!npm install -g @biomejs/biome
!pip install -qU ruff codespell basedpyright

In [ ]:
import time
import subprocess


def run_cmd(cmd):
  print(f"\n[EXEC] {cmd}")
  result = subprocess.run(
    cmd, shell=True, cwd=MONOREPO_DIR, capture_output=True, text=True
  )
  if result.stdout:
    print(result.stdout.strip()[:500] + ("..." if len(result.stdout) > 500 else ""))
  if result.stderr:
    print(f"ERRORS:\n{result.stderr.strip()[:500]}")


print("☁️ Continuous Linter/Refactor Worker Online. Shield is up.")

while True:
  try:
    # 1. Biome: Format and safe-lint JavaScript/TypeScript/JSON
    run_cmd("biome check --write .")

    # 2. Ruff: Format and safe-lint Python (auto-fixing syntax/imports)
    run_cmd("ruff check --fix .")
    run_cmd("ruff format .")

    # 3. Codespell: Remove spelling errors across the repo
    run_cmd(
      "codespell --write-changes --skip='*.json,*.lock,*.webp,*.sqlite,*.csv,*.tmp,*.git*' "
    )

    # Sleep for a longer period to avoid destroying Google Drive IOPS
    print("\n[Colab Linter] Pass complete. Sleeping for 2 minutes...\n")
    time.sleep(120)

  except KeyboardInterrupt:
    print("\n🛑 Linter Worker Safely Terminated.")
    break
  except Exception as e:
    print(f"⚠️ Loop failure: {e}")
    time.sleep(10)